# 05 — LLM Dev Analysis

Analysis of LLM-based grading on the **DEV split** (50 responses) for two prompting strategies:
- `direct_v1`: standalone grader returning `{reasoning, score}`
- `rubric_v1`: criterion-aware grader with 5 sub-scores

Sections:
1. Metrics (MAE, RMSE, Pearson, Spearman, QWK, Within-1) and parse failures
2. Mean signed error (bias: lenient vs strict)
3. Confusion matrix (rounded human vs rounded predicted)
4. Top-5 largest disagreements
5. Rubric: per-criterion mean and correlation with human score
6. "NO ANSWER" / empty response counts

> **Note**: This notebook operates only on the DEV split. Test split is not evaluated here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

from src.metrics import evaluate
from src.llm_eval import RUBRIC_CRITERIA

# ── Paths ─────────────────────────────────────────────────────────────────────
DIRECT_PATH = Path('../results/llm_direct_dev.csv')
RUBRIC_PATH = Path('../results/llm_rubric_dev.csv')
FIG_DIR = Path('../results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
PALETTE = {'direct': '#4C9BE8', 'rubric': '#E87D4C'}

def load(path):
    df = pd.read_csv(path)
    # Only include rows with valid (non-NaN) pred_score
    df['pred_score'] = pd.to_numeric(df['pred_score'], errors='coerce')
    return df

print(f'Direct path exists: {DIRECT_PATH.exists()}')
print(f'Rubric path exists: {RUBRIC_PATH.exists()}')

In [ ]:
direct = load(DIRECT_PATH)
rubric = load(RUBRIC_PATH)

print(f'Direct rows : {len(direct)}')
print(f'Rubric rows : {len(rubric)}')
print()
print('Direct columns :', direct.columns.tolist())
print('Rubric columns :', rubric.columns.tolist())

---
## 1. Metrics on Dev — MAE, RMSE, Pearson, Spearman, QWK, Within-1

In [ ]:
def compute_metrics_for_df(df, label):
    valid = df.dropna(subset=['pred_score', 'human_score'])
    n_valid = len(valid)
    n_total = len(df)
    n_parse_fail = int((~df['parse_ok'].astype(bool)).sum())

    if n_valid == 0:
        print(f"{label}: No valid predictions!")
        return None

    m = evaluate(valid['human_score'].values, valid['pred_score'].values)
    m['n_valid'] = n_valid
    m['n_total'] = n_total
    m['parse_failures'] = n_parse_fail
    m['method'] = label
    return m

m_direct = compute_metrics_for_df(direct, 'direct_v1')
m_rubric = compute_metrics_for_df(rubric, 'rubric_v1')

metrics_rows = []
for m in [m_direct, m_rubric]:
    if m:
        metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows).set_index('method')

display_cols = ['mae', 'rmse', 'pearson', 'spearman', 'qwk', 'within_1', 'parse_failures', 'n_valid', 'n_total']
print('\n=== DEV METRICS ===')
print(metrics_df[display_cols].to_string(float_format='{:.4f}'.format))

In [ ]:
# Bar chart comparing key metrics
metric_names = ['mae', 'rmse', 'pearson', 'spearman', 'qwk', 'within_1']
metric_labels = ['MAE ↓', 'RMSE ↓', 'Pearson ↑', 'Spearman ↑', 'QWK ↑', 'Within-1 ↑']

fig, axes = plt.subplots(1, len(metric_names), figsize=(14, 3.5))
methods = metrics_df.index.tolist()
colors = [PALETTE.get(m.split('_')[0], '#888') for m in methods]

for ax, col, lbl in zip(axes, metric_names, metric_labels):
    vals = metrics_df[col].values
    bars = ax.bar(methods, vals, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(lbl, fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(['Direct', 'Rubric'], fontsize=9)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('LLM Grading — DEV Split Metrics (Direct vs Rubric)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'llm_dev_metrics_bar.png', bbox_inches='tight')
plt.show()
print('Saved: results/figures/llm_dev_metrics_bar.png')

---
## 2. Mean Signed Error — Is the Model Lenient or Strict?

In [ ]:
def signed_error_analysis(df, label):
    valid = df.dropna(subset=['pred_score', 'human_score']).copy()
    valid['signed_error'] = valid['pred_score'] - valid['human_score']
    mse = valid['signed_error'].mean()
    std = valid['signed_error'].std()
    direction = 'LENIENT (over-predicts)' if mse > 0.05 else ('STRICT (under-predicts)' if mse < -0.05 else 'WELL-CALIBRATED')
    print(f'{label}:')
    print(f'  Mean signed error : {mse:+.4f}')
    print(f'  Std signed error  : {std:.4f}')
    print(f'  Interpretation    : {direction}')
    print()
    return valid

direct_valid = signed_error_analysis(direct, 'direct_v1')
rubric_valid = signed_error_analysis(rubric, 'rubric_v1')

In [ ]:
# Signed error distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)

for ax, (df_v, label, color) in zip(axes, [
    (direct_valid, 'Direct (direct_v1)', PALETTE['direct']),
    (rubric_valid, 'Rubric (rubric_v1)', PALETTE['rubric'])
]):
    ax.hist(df_v['signed_error'], bins=15, color=color, edgecolor='white', alpha=0.85)
    mse = df_v['signed_error'].mean()
    ax.axvline(mse, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mse:+.3f}')
    ax.axvline(0, color='grey', linestyle=':', linewidth=1)
    ax.set_xlabel('Predicted − Human Score', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Signed Error Distribution — DEV Split', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'llm_dev_signed_error.png', bbox_inches='tight')
plt.show()
print('Saved: results/figures/llm_dev_signed_error.png')

---
## 3. Confusion Matrix — Rounded Human vs Rounded Predicted

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion(df, label, color, ax):
    valid = df.dropna(subset=['pred_score', 'human_score'])
    h_int = valid['human_score'].round().clip(0, 5).astype(int)
    p_int = valid['pred_score'].round().clip(0, 5).astype(int)
    labels = list(range(0, 6))
    cm = confusion_matrix(h_int, p_int, labels=labels)
    
    # Row-normalise for readability
    with np.errstate(divide='ignore', invalid='ignore'):
        cm_norm = np.where(cm.sum(axis=1, keepdims=True) > 0,
                           cm / cm.sum(axis=1, keepdims=True),
                           0)
    sns.heatmap(
        cm_norm, annot=cm, fmt='d', cmap='Blues',
        linewidths=0.4, linecolor='white',
        xticklabels=labels, yticklabels=labels,
        ax=ax, cbar=False, vmin=0, vmax=1
    )
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Score (rounded)', fontsize=9)
    ax.set_ylabel('Human Score (rounded)', fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_confusion(direct, 'Direct (direct_v1)', PALETTE['direct'], axes[0])
plot_confusion(rubric, 'Rubric (rubric_v1)', PALETTE['rubric'], axes[1])
fig.suptitle('Confusion Matrix — Rounded Scores (DEV Split)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'llm_dev_confusion.png', bbox_inches='tight')
plt.show()
print('Saved: results/figures/llm_dev_confusion.png')

---
## 4. Top-5 Largest Disagreements

In [ ]:
# Load the dev split to get question text and reference/student answers
dev_data = pd.read_csv('../data/processed/dev.csv')

def top_disagreements(eval_df, dev_df, label, n=5):
    valid = eval_df.dropna(subset=['pred_score', 'human_score']).copy()
    valid['abs_error'] = (valid['pred_score'] - valid['human_score']).abs()
    top = valid.nlargest(n, 'abs_error').reset_index(drop=True)

    print(f'\n=== {label} — Top-{n} Largest Disagreements ===')
    for _, row in top.iterrows():
        qid = row['question_id']
        # Match by question_id
        dev_row = dev_df[dev_df['question_id'] == qid]
        if dev_row.empty:
            # Fallback: try by position if qid is integer index
            try:
                dev_row = dev_df.iloc[[int(qid)]]
            except Exception:
                dev_row = pd.DataFrame()

        question = dev_row['question'].values[0] if not dev_row.empty else 'N/A'
        reference = dev_row['reference_answer'].values[0] if not dev_row.empty else 'N/A'
        student = dev_row['student_answer'].values[0] if not dev_row.empty else 'N/A'

        print(f"\n  QID: {qid}  |  Human: {row['human_score']:.1f}  |  LLM: {row['pred_score']:.1f}  |  Δ: {row['pred_score'] - row['human_score']:+.1f}")
        print(f"  Q: {str(question)[:120]}")
        print(f"  Ref: {str(reference)[:120]}")
        print(f"  Student: {str(student)[:120]}")
        print(f"  Reasoning: {str(row.get('reasoning', ''))[:200]}")
    return top

top_direct = top_disagreements(direct, dev_data, 'Direct (direct_v1)')
top_rubric = top_disagreements(rubric, dev_data, 'Rubric (rubric_v1)')

---
## 5. Rubric — Per-Criterion Means and Correlations with Human Score

In [ ]:
from scipy.stats import pearsonr
import warnings

crit_cols = [f'score_{c}' for c in RUBRIC_CRITERIA]
available = [c for c in crit_cols if c in rubric.columns]

rubric_valid_all = rubric.dropna(subset=['human_score', 'pred_score'])

print('=== Rubric Criterion Analysis ===')
crit_stats = []
for crit, col in zip(RUBRIC_CRITERIA, available):
    vals = pd.to_numeric(rubric_valid_all[col], errors='coerce')
    h = rubric_valid_all['human_score'].values
    v = vals.values

    # Only correlate non-NaN pairs
    mask = ~np.isnan(v)
    if mask.sum() > 2:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            corr, _ = pearsonr(h[mask], v[mask])
    else:
        corr = np.nan

    row = {
        'criterion': crit,
        'mean': float(np.nanmean(v)),
        'std': float(np.nanstd(v)),
        'pearson_with_human': corr,
        'n_valid': int(mask.sum())
    }
    crit_stats.append(row)
    print(f"  {crit:15s}  mean={row['mean']:.3f}  std={row['std']:.3f}  r_with_human={corr:.3f}  n={row['n_valid']}")

crit_df = pd.DataFrame(crit_stats).set_index('criterion')
print()
print('Overall rubric score (mean overall):', rubric_valid_all['pred_score'].mean())
print('Human score (mean):', rubric_valid_all['human_score'].mean())

In [ ]:
# Plot criterion means and correlations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# -- Mean scores per criterion --
ax = axes[0]
y_pos = range(len(RUBRIC_CRITERIA))
means = crit_df['mean'].values
stds = crit_df['std'].values
bars = ax.barh(y_pos, means, xerr=stds, color=PALETTE['rubric'],
               alpha=0.8, edgecolor='white', capsize=4)
ax.set_yticks(y_pos)
ax.set_yticklabels([c.capitalize() for c in RUBRIC_CRITERIA], fontsize=10)
ax.set_xlabel('Mean Score (0–5)', fontsize=10)
ax.set_title('Mean Criterion Scores ± SD', fontsize=11, fontweight='bold')
ax.set_xlim(0, 5.5)
for bar, v in zip(bars, means):
    ax.text(v + 0.05, bar.get_y() + bar.get_height()/2, f'{v:.2f}', va='center', fontsize=9)

# -- Correlation with human score --
ax2 = axes[1]
corrs = crit_df['pearson_with_human'].values
colors_corr = [PALETTE['rubric'] if c >= 0 else '#E84C4C' for c in corrs]
bars2 = ax2.barh(y_pos, corrs, color=colors_corr, alpha=0.85, edgecolor='white')
ax2.set_yticks(y_pos)
ax2.set_yticklabels([c.capitalize() for c in RUBRIC_CRITERIA], fontsize=10)
ax2.set_xlabel('Pearson r with Human Score', fontsize=10)
ax2.set_title('Criterion–Human Score Correlation', fontsize=11, fontweight='bold')
ax2.axvline(0, color='grey', linestyle=':', linewidth=1)
for bar, v in zip(bars2, corrs):
    offset = 0.02 if v >= 0 else -0.08
    ax2.text(v + offset, bar.get_y() + bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)

fig.suptitle('Rubric Criterion Analysis — DEV Split', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'llm_dev_rubric_criteria.png', bbox_inches='tight')
plt.show()
print('Saved: results/figures/llm_dev_rubric_criteria.png')

---
## 6. "NO ANSWER" / Empty Response Counts

In [ ]:
dev_data = pd.read_csv('../data/processed/dev.csv')

# Define "NO ANSWER" as empty / whitespace-only / common no-answer phrases
NO_ANSWER_PATTERNS = [
    r'^\s*$',
    r'^\s*(no\s+answer|n/a|not\s+answered?|no\s+response|blank)\s*$'
]

import re

def is_no_answer(text):
    if not isinstance(text, str):
        return True
    for pat in NO_ANSWER_PATTERNS:
        if re.match(pat, text.strip(), re.IGNORECASE):
            return True
    return False

dev_data['is_no_answer'] = dev_data['student_answer'].apply(is_no_answer)
n_no_answer = dev_data['is_no_answer'].sum()

print(f'Total dev responses : {len(dev_data)}')
print(f'"NO ANSWER" / empty : {n_no_answer} ({100*n_no_answer/len(dev_data):.1f}%)')
print()

if n_no_answer > 0:
    print('Rows with no answer:')
    for _, r in dev_data[dev_data['is_no_answer']].iterrows():
        print(f"  QID={r['question_id']}  human_score={r['human_score']}  student_answer={repr(str(r['student_answer'])[:80])}")
else:
    print('No empty or "NO ANSWER" responses detected in the dev set.')

In [ ]:
# Also check how LLM handled zero-score items (human_score == 0)
print('=== Zero-Score Items (human_score == 0) ===')
for label, df in [('direct', direct), ('rubric', rubric)]:
    zero = df[df['human_score'] == 0].copy()
    print(f'{label}: {len(zero)} items with human_score=0')
    if len(zero) > 0:
        valid_zero = zero.dropna(subset=['pred_score'])
        print(f'  Mean LLM pred on zero-score items: {valid_zero["pred_score"].mean():.3f}')
        print(f'  LLM also scored 0: {(valid_zero["pred_score"].round() == 0).sum()} / {len(valid_zero)}')

---
## Summary Table

In [ ]:
print('\n' + '='*65)
print('LLM DEV ANALYSIS SUMMARY')
print('='*65)
print(f'{"Metric":<20} {"Direct":>10} {"Rubric":>10}')
print('-'*42)

for col, lbl in [
    ('mae', 'MAE ↓'),
    ('rmse', 'RMSE ↓'),
    ('pearson', 'Pearson ↑'),
    ('spearman', 'Spearman ↑'),
    ('qwk', 'QWK ↑'),
    ('within_1', 'Within-1 ↑'),
]:
    d_val = m_direct[col] if m_direct else float('nan')
    r_val = m_rubric[col] if m_rubric else float('nan')
    print(f'{lbl:<20} {d_val:>10.4f} {r_val:>10.4f}')

print('-'*42)
d_pf = m_direct['parse_failures'] if m_direct else 'N/A'
r_pf = m_rubric['parse_failures'] if m_rubric else 'N/A'
print(f'{"Parse failures":<20} {str(d_pf):>10} {str(r_pf):>10}')
print('='*65)

# Bias
for label, df_v in [('Direct', direct_valid), ('Rubric', rubric_valid)]:
    mse = df_v['signed_error'].mean()
    direction = 'lenient' if mse > 0.05 else ('strict' if mse < -0.05 else 'well-calibrated')
    print(f'{label} bias: {mse:+.4f} ({direction})')

print(f'\nNO ANSWER responses in dev: {n_no_answer} / {len(dev_data)}')